# Assignment 5 — Whitted-Style Ray Tracing

> **GAMES101 — Intro to Computer Graphics** (Lingqi Yan, UCSB).
> Course site: <https://sites.cs.ucsb.edu/~lingqi/teaching/games101.html>
>
> The course ships C++ starter code with Eigen + OpenCV. I'm doing the same tasks in
> Python notebooks so I can iterate on the math cell-by-cell. Notes at the top of each
> notebook are what I actually needed to remember to get the assignment out.

## Topic

First real jump out of rasterisation. **Ray tracing** flips the pipeline: instead
of pushing triangles to pixels, we push rays from each pixel out into the world and
ask what they hit.

**Whitted-style** means: primary rays for visibility, plus mirror-reflection and
refraction bounces for glass/mirrors. It gives you sharp reflections and refractions
on smooth surfaces, but no soft shadows or diffuse interreflection (that's A7).

The two functions this assignment asks for:
1. **Primary ray generation** — map `(pixel_i, pixel_j)` → world-space direction.
   Aspect ratio and Y flip bite here.
2. **Möller–Trumbore** — closed-form ray/triangle intersection using barycentrics.

![recursive raytrace](https://upload.wikimedia.org/wikipedia/commons/3/32/Recursive_raytrace_of_a_sphere.png)
*Whitted-style: reflection + refraction on a glass sphere (Wikipedia).*

See: [Ray tracing (graphics)](https://en.wikipedia.org/wiki/Ray_tracing_(graphics)), [Möller–Trumbore intersection algorithm](https://en.wikipedia.org/wiki/M%C3%B6ller%E2%80%93Trumbore_intersection_algorithm).


In [1]:
import numpy as np

def gen_ray(i, j, W, H, fov_deg):
    # naive: forgot NDC remap and aspect
    scale = np.tan(np.deg2rad(fov_deg) / 2)
    x = (2 * i / W - 1) * scale
    y = (2 * j / H - 1) * scale
    d = np.array([x, y, -1.0])
    return d / np.linalg.norm(d)


In [2]:
def ray_triangle(orig, dir_, v0, v1, v2):
    e1 = v1 - v0
    e2 = v2 - v0
    p = np.cross(dir_, e2)
    det = e1 @ p
    if abs(det) < 1e-8:
        return None
    inv = 1.0 / det
    s = orig - v0
    u = (s @ p) * inv
    if u < 0 or u > 1:
        return None
    q = np.cross(s, e1)
    v = (dir_ @ q) * inv
    if v < 0 or u + v > 1:
        return None
    t = (e2 @ q) * inv
    return t if t > 0 else None


Image is stretched -- I forgot to multiply x by aspect ratio and to flip y (screen space vs. world y).